In [1]:
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql.functions import col, to_timestamp, unix_timestamp, when, lit, current_timestamp, coalesce, regexp_replace, length, lower
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.functions import regexp_replace, trim, concat_ws, to_date, least, greatest, current_date, udf

In [2]:
# Initialize Spark session with legacy time parser
spark = SparkSession.builder \
    .appName("GOLD Layer Creation") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.eventLog.logBlockUpdates.enabled", True)\
    .enableHiveSupport() \
    .getOrCreate()

KeyboardInterrupt: 

In [3]:
ports = spark.sql("SELECT * FROM SILVER.ports")
# spark.sql("SELECT * FROM SILVER.ports")

In [4]:
vessels = spark.sql("SELECT * FROM SILVER.vessels")

In [5]:
# seismic = spark.sql("SELECT * FROM SILVER.vessels")

In [8]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
spark.sql("use gold")

""


### Gold Views based on silver.vessels

In [9]:
#1. Distinct Vessels Tracked

query = """
CREATE OR REPLACE VIEW gold.vessel_count AS
SELECT COUNT(DISTINCT name) AS distinct_vessel_count
FROM silver.vessels;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_count")

distinct_vessel_count
101


In [10]:
# 2. Vessel Movements This Week / Month
query = """CREATE OR REPLACE VIEW gold.vessel_movements_summary AS
SELECT
    COUNT(*) AS total_movements,
    COUNT(CASE WHEN arrival_date >= date_trunc('week', current_date) THEN 1 END) AS this_week,
    COUNT(CASE WHEN arrival_date >= date_trunc('month', current_date) THEN 1 END) AS this_month
FROM silver.vessels;
"""

spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_movements_summary")

total_movements,this_week,this_month
651,617,649


In [11]:
# 3. Top 5 Vessel Types by Count
query = """
CREATE OR REPLACE VIEW gold.top_5_vessel_types AS
SELECT
    type,
    COUNT(*) AS vessel_count
FROM silver.vessels
GROUP BY type
ORDER BY vessel_count DESC
LIMIT 5;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_5_vessel_types")

type,vessel_count
-,491
Tanker,91
Cargo ship,49
Tanker (HAZ-A),8
Other type,7


In [12]:
# 4. Average Deadweight and Gross Tonnage per Vessel Type
query = """
CREATE OR REPLACE VIEW gold.avg_tonnage_by_vessel_type AS
SELECT
    type,
    AVG(deadweight) AS avg_deadweight,
    AVG(gross_tonnage) AS avg_gross_tonnage
FROM silver.vessels
GROUP BY type;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.avg_tonnage_by_vessel_type")

type,avg_deadweight,avg_gross_tonnage
Tanker,22191.134615384617,12364.326923076924
Other type,3220.0,2042.0
Tanker (HAZ-A),52552.0,29097.5
Cargo ship (HAZ-A),12720.0,10308.0
Tanker (HAZ-B),105778.0,49838.0
-,5396.91371681416,3289.880530973451
Cargo ship,26699.061224489797,19513.020408163266


In [13]:
# 5. Vessels Built Before 2000
query = """
CREATE OR REPLACE VIEW gold.vessels_built_before_2000 AS
SELECT
    COUNT(*) AS vessel_count_pre_2000
FROM silver.vessels
WHERE year_built < 2000;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessels_built_before_2000")

vessel_count_pre_2000
380


In [14]:
# 6. Vessels per Country (based on destination port country)
query = """
CREATE OR REPLACE VIEW gold.vessels_by_destination_country AS
SELECT
    destination_port_country,
    COUNT(*) AS vessel_count
FROM silver.vessels
GROUP BY destination_port_country;
"""

spark.sql(query)
spark.sql("SELECT * FROM gold.vessels_by_destination_country")

destination_port_country,vessel_count
Yemen,2
Turkey,2
Iraq,1
null,136
India,4
Spain,2
Bangladesh,4
-,491
Saudi Arabia,1
Egypt,8


In [15]:
# 7. Vessels per Port in Last 30 Days
query = """
CREATE OR REPLACE VIEW gold.recent_visits_per_port AS
SELECT
    destination_port_name,
    COUNT(*) AS vessel_visits
FROM silver.vessels
WHERE arrival_date >= current_date - INTERVAL 30 DAYS
GROUP BY destination_port_name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.recent_visits_per_port")

destination_port_name,vessel_visits
Aliaga,1
EGACT,1
BARCARENA - BRAZIL,1
Hazira,4
Wadi Feiran,1
RAS SADAT,2
Suez Canal,1
ZET_BAY EG,1
Jeddah,1
El Dekheila,1


In [16]:
# 8. Most Used Vessel Routes
query = """
CREATE OR REPLACE VIEW gold.top_vessel_routes AS
SELECT
    last_port_name,
    destination_port_name,
    COUNT(*) AS route_count
FROM silver.vessels
GROUP BY last_port_name, destination_port_name
ORDER BY route_count DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_vessel_routes")

last_port_name,destination_port_name,route_count
-,-,491
As Suways / Suez ...,Destination not a...,35
Alexandria,Destination not a...,15
Port Said,Destination not a...,15
Port Tewfik,Destination not a...,12
Ismailia Anch.,Destination not a...,7
Port Said Anch.,Destination not a...,7
Safaga,SUEZ-SAFAGA,6
Jeddah Anchorage,JEDDAH<>SAWAKIN,5
Singapore Anch. 4,Hazira,4


In [17]:
# 9. Average Arrival/Departure Time per Port
query = """
CREATE OR REPLACE VIEW gold.avg_stay_duration_by_port AS
SELECT
    destination_port_name,
    AVG(abs(UNIX_TIMESTAMP(arrival_date) - UNIX_TIMESTAMP(departure_date))) / 3600 AS avg_duration_hours
FROM silver.vessels
WHERE arrival_date IS NOT NULL AND departure_date IS NOT NULL
GROUP BY destination_port_name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.avg_stay_duration_by_port")

destination_port_name,avg_duration_hours
Aliaga,24.0
EGACT,24.0
BARCARENA - BRAZIL,264.0
Hazira,0.0
Wadi Feiran,624.0
RAS SADAT,0.0
ZET_BAY EG,432.0
Suez Canal,0.0
Jeddah,432.0
El Dekheila,48.0


In [18]:
# 10. Vessel Classes by Highest Average Gross Tonnage
query = """
CREATE OR REPLACE VIEW gold.vessel_class_by_avg_tonnage AS
SELECT
    type AS vessel_class,
    AVG(gross_tonnage) AS avg_gross_tonnage
FROM silver.vessels
GROUP BY vessel_class
ORDER BY avg_gross_tonnage DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.vessel_class_by_avg_tonnage")

vessel_class,avg_gross_tonnage
Tanker (HAZ-B),49838.0
Tanker (HAZ-A),29097.5
Cargo ship,19513.020408163266
Tanker,12364.326923076924
Cargo ship (HAZ-A),10308.0
-,3289.880530973451
Other type,2042.0


In [19]:
# 11. Top 10 Busiest Routes in Last 30 Days

query = """
CREATE OR REPLACE VIEW gold.top_routes_last_30_days AS
SELECT
    last_port_name,
    destination_port_name,
    COUNT(*) AS voyage_count
FROM silver.vessels
WHERE arrival_date >= current_date - INTERVAL 30 DAYS
GROUP BY last_port_name, destination_port_name
ORDER BY voyage_count DESC;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.top_routes_last_30_days limit 10")

last_port_name,destination_port_name,voyage_count
-,-,491
As Suways / Suez ...,Destination not a...,35
Port Said,Destination not a...,15
Alexandria,Destination not a...,15
Port Tewfik,Destination not a...,12
Ismailia Anch.,Destination not a...,7
Port Said Anch.,Destination not a...,7
Safaga,SUEZ-SAFAGA,6
Jeddah Anchorage,JEDDAH<>SAWAKIN,5
Singapore Anch. 4,Hazira,4


In [ ]:
# Feel Free to add new views or qeustions with answer about vessels


### Gold Views based on silver.ports

In [20]:
# 12. Distinct Ports Count
query = """
CREATE OR REPLACE VIEW gold.distinct_ports_count AS
SELECT COUNT(DISTINCT OID_) AS total_ports
FROM silver.ports;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.distinct_ports_count")
# The output is 0 because all OID_ values are null

total_ports
0


In [21]:
# 13. Ports by Country or Region
query = """
CREATE OR REPLACE VIEW gold.port_distribution_by_region AS
SELECT
    Country_Code,
    Region_Name,
    COUNT(*) AS port_count
FROM silver.ports
GROUP BY Country_Code, Region_Name;
"""
spark.sql(query)
spark.sql("SELECT * FROM gold.port_distribution_by_region")

Country_Code,Region_Name,port_count
Canada,Canada Lake Ontar...,44
United Kingdom,England South Coa...,56
Turkey,Turkey Europe -- ...,20
Georgia,Georgia -- 44275,6
Wallis and Futuna,Iles Wallis -- 55640,2
Thailand,Thailand -- 49760,10
U.S. Virgin Islands,St Thomas -- 11305,2
Russia,Russia -- 28670,4
Greece,Greece East Coast...,44
South Georgia and...,South Georgia -- ...,6


In [ ]:
# Feel Free to add new views or qeustions with answer about ports


## Seismic

In [53]:
query = """
SELECT *
FROM silver.seismic
LIMIT 3
"""
spark.sql(query)

PK,UNID,SOURCE_ID,SOURCE_CATALOG,LASTUPDATE_UTC,EVENT_TIME_UTC,FLYNN_REGION,LAT,LON,DEPTH,EVTYPE,AUTH,MAG,MAGTYPE,ACTION,RECEIVED_AT_UTC,SOURCE_DB,SOURCE_TABLE,OPERATION,BPK,LOAD_At
801,20250713_0000025,1831472,EMSC-RT,2025-07-13 02:35:...,2025-07-13 01:49:58,CENTRAL TURKEY,38.1542,36.8469,10.2,ke,AFAD,1.2,ml,create,2025-07-13 01:49:58,maritime_logistics,seismic_events,u,795,2025-07-19
802,20250713_0000028,1831475,EMSC-RT,2025-07-13 02:45:...,2025-07-13 02:33:30,NEAR EAST COAST O...,39.9,142.4,10.0,ke,JMA,3.8,m,create,2025-07-13 02:33:30,maritime_logistics,seismic_events,u,6,2025-07-19
803,20250713_0000028,1831475,EMSC-RTS,2025-07-13 02:45:...,2025-07-13 02:33:30,NEAR EAST COAST O...,39.9,142.4,10.0,ke,JMA,3.8,m,create,2025-07-13 02:33:30,maritime_logistics,seismic_events,u,802,2025-07-19


In [ ]:
# Feel Free to add new views or qeustions with answer about seismic


In [29]:
# Get Seismic which is Nearset to port within x kilometers
query = """
SELECT S.UNID, S.LAT, S.LON
FROM silver.seismic S
WHERE ACTION = 'create'
LIMIT 3
"""
spark.sql(query)

UNID,LAT,LON
20250713_0000025,38.1542,36.8469
20250713_0000028,39.9,142.4
20250713_0000028,39.9,142.4


In [30]:
query = """
SELECT P.Main_Port_Name, P.Region_Name, P.Country_Code, P.Latitude, P.Longitude
FROM silver.ports P
LIMIT 3
"""
spark.sql(query)

Main_Port_Name,Region_Name,Country_Code,Latitude,Longitude
Maurer,United States E C...,United States,40.533333,-74.25
Iharana,Madagascar -- 47350,Madagascar,-13.35,50.0
Chake Chake,Tanzania -- 46965,Tanzania,-5.25,39.766667


In [61]:
# Create or replace gold table with filtered results
query = """
CREATE OR REPLACE VIEW port_seismic_within_50km AS 
SELECT 
    P.Main_Port_Name,
    P.Region_Name,
    P.Country_Code,
    P.Latitude AS port_lat,
    P.Longitude AS port_lon,
    S.UNID,
    S.LAT AS seismic_lat,
    S.LON AS seismic_lon,
    6371 * 2 * ASIN(SQRT(
        POWER(SIN(RADIANS((S.LAT - P.Latitude)/2)), 2) +
        COS(RADIANS(P.Latitude)) * COS(RADIANS(S.LAT)) *
        POWER(SIN(RADIANS((S.LON - P.Longitude)/2)), 2)
    )) AS distance_km
FROM silver.ports P
JOIN silver.seismic S
ON (
    6371 * 2 * ASIN(SQRT(
        POWER(SIN(RADIANS((S.LAT - P.Latitude)/2)), 2) +
        COS(RADIANS(P.Latitude)) * COS(RADIANS(S.LAT)) *
        POWER(SIN(RADIANS((S.LON - P.Longitude)/2)), 2)
    )) < 50
)
WHERE S.ACTION = 'create' AND S.MAG > 4.0
"""
spark.sql(query)

""


In [ ]:
# Which ports are within 100 km of high-magnitude seismic events
query = """
SELECT Main_Port_Name, COUNT(Main_Port_Name) AS CNT
FROM port_seismic_within_50km
GROUP BY Main_Port_Name
ORDER BY CNT desc
"""

spark.sql(query)

In [ ]:
# Feel Free to add new views or qeustions with answer


In [63]:
spark.stop()